# scRNA-seq: major cell types (Fig. 2B, 2C, 2H; Supplementary Fig. S6A–C, S6F)
B16F10 footpad tumors from mice on a soft mattress vs. intermittent forced wheel running (14 days).
- **Fig. 2B** UMAP of major cell types
- **Fig. 2C** cell type proportion per group (final bar plot made in Prism)
- **Fig. 2H** EMT-like and CPRR signature scores in melanoma cells
- **Supplementary Fig. S6A–C** UMAP by group, cell type proportion per sample, marker gene dot plot
- **Supplementary Fig. S6F** signatures increased in melanoma cells from the Running group

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

from sc_utils import toCount_HVG_Scale

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.logging.print_header()
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')

DATA_DIR = '../data'   # Melanoma_scRNAseq_data.h5ad (GEO: GSE289587 / Zenodo)

In [ ]:
adata = sc.read(f"{DATA_DIR}/Melanoma_scRNAseq_data.h5ad")
adata

## Fig. 2B — UMAP
Cell type labels were added in Adobe Illustrator.

In [ ]:
sc.pl.umap(adata, color = 'Level1_annotation', frameon = False, title = '')

## Fig. 2C — Cell type proportion
Final bar plot was generated with Prism.

In [ ]:
proportions = adata.obs.groupby(['sample', 'Level1_annotation']).size().unstack(fill_value=0)
proportions = proportions.div(proportions.sum(axis=1), axis=0)
proportions = proportions * 100

In [ ]:
cell_type_order = ['Melanoma', 'Macrophage', 'Fibroblast', 'Neutrophil', 'B', 'DC', 'T', 'Endothelial', 'Monocyte', 'Pericyte', 'NK']
proportions = proportions[cell_type_order]

In [ ]:
proportions_with_group = proportions.reset_index()
proportions_with_group['group'] = proportions_with_group['sample'].map({
    'S1': 'Mattress', 'S2': 'Mattress', 'Sponge_1': 'Mattress', 'Sponge_3': 'Mattress',
    'T1': 'Running', 'T2': 'Running', 'Tread_1': 'Running', 'Tread_2': 'Running', 'Tread_3': 'Running'
})

In [ ]:
melted_data = pd.melt(proportions_with_group, id_vars=['sample', 'group'], var_name='cell_type', value_name='proportion')
melted_data['cell_type'] = pd.Categorical(melted_data['cell_type'], categories=cell_type_order, ordered=True)

In [ ]:
plt.figure(figsize=(20, 10))

g = sns.barplot(
    data=melted_data,
    x="cell_type",
    y="proportion",
    hue="group",
    palette=['#D4D4D4CC', '#0000FFCC'],
    width=0.8
)

g.grid(False)

g.spines['left'].set_color('black')
g.spines['bottom'].set_color('black')
g.spines['left'].set_linewidth(2)
g.spines['bottom'].set_linewidth(2)

g.spines['top'].set_visible(False)
g.spines['right'].set_visible(False)

plt.xticks(rotation=60, fontsize=20)

plt.yticks(fontsize=20)

plt.ylabel("Cell type proportion (%)", fontsize=24)
plt.xlabel("", fontsize=24)

plt.legend(title="Group", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20)
plt.setp(g.get_legend().get_title(), fontsize=20)

plt.tight_layout()
plt.show()

## Supplementary Fig. S6A — UMAP by experimental group
The colours are scanpy's default colours for Sponge (Mattress) and Tread (Running), as in the original figure.

In [ ]:
adata.obs['group_name'] = adata.obs['group'].map({'Sponge': 'Mattress', 'Tread': 'Running'}).astype('category')
adata.uns['group_name_colors'] = ['#1f77b4', '#ff7f0e']   # Mattress, Running
print(adata.obs['group_name'].value_counts())
sc.pl.umap(adata, color='group_name', frameon=False, title='')

## Supplementary Fig. S6B — Cell type proportion per sample
Samples are numbered in sorted order (S1, S2, Sponge_1, Sponge_3 = Mattress1–4; T1, T2, Tread_1–3 = Running1–5);
the number of cells per sample is given in parentheses.

In [ ]:
cell_type_order = ['Melanoma', 'Macrophage', 'Fibroblast', 'Neutrophil', 'B', 'DC', 'T', 'Endothelial',
                   'Monocyte', 'Pericyte', 'NK']
legend_names = {'B': 'B cell', 'DC': 'Dendritic cell', 'T': 'T cell', 'Endothelial': 'Endothelial cell', 'NK': 'NK cells'}
level1_colors = dict(zip(adata.obs['Level1_annotation'].cat.categories, adata.uns['Level1_annotation_colors']))

counts = adata.obs.groupby(['sample', 'Level1_annotation'], observed=False).size().unstack(fill_value=0)
total_cells = counts.sum(axis=1)
ratios = (counts.div(total_cells, axis=0) * 100).reindex(columns=cell_type_order, fill_value=0)
ratios.index = [f'Mattress{i + 1}\n({total_cells[s]})' if i < 4 else f'Running{i - 3}\n({total_cells[s]})'
                for i, s in enumerate(ratios.index)]
ratios.columns = [legend_names.get(c, c) for c in ratios.columns]

ax = ratios.plot(kind='bar', stacked=True, color=[level1_colors[c] for c in cell_type_order], figsize=(8, 5),
                 edgecolor='black', linewidth=0.5, width=0.7)
ax.set_ylabel('Cell type proportion (%)')
ax.set_xlabel('')
ax.set_ylim(0, 100)
plt.xticks(rotation=45, ha='center')
ax.legend(title='Cell type', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
ax.grid(False)
ax.figure.set_label('celltype_proportion_per_sample')
plt.show()

## Supplementary Fig. S6C — Cell type marker genes

In [ ]:
sc.pl.dotplot(adata, ['Cd19', 'Xcr1', 'Pecam1', 'Pdgfra', 'C1qc', 'Mlana', 'Cd14', 'Nkg7', 'S100a9', 'Rgs5', 'Cd3e'],
              groupby='Level1_annotation', standard_scale='var')

## Fig. 2H — EMT-like and CPRR signatures in melanoma cells
Scores from `sc.tl.score_genes` on log-normalized expression (`adata.raw`): Hallmark EMT (EMT-like) and
GO:BP cytoplasmic pattern recognition receptor signaling pathway (CPRR).
The manuscript panels were drawn with ggplot2 (`geom_violin(draw_quantiles = 0.5)`); `plot_group_violin` reproduces them
(density, violin width and the median line computed as in ggplot2).

In [ ]:
import json
from scipy.stats import ttest_ind, mannwhitneyu
from sc_utils import plot_group_violin

with open(f'{DATA_DIR}/genesets_used.json', 'r') as f:
    genesets = json.load(f)

Mel = adata[adata.obs['Level1_annotation'] == 'Melanoma'].copy()
Mel.obs['group'] = Mel.obs['group'].map({'Sponge': 'Mattress', 'Tread': 'Running'})
sc.tl.score_genes(Mel, genesets['HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION'], score_name='EMT_like_score')
sc.tl.score_genes(Mel, genesets['GOBP_CYTOPLASMIC_PATTERN_RECOGNITION_RECEPTOR_SIGNALING_PATHWAY'], score_name='CPRR_score')

In [ ]:
for score, title in [('EMT_like_score', 'EMT-like signature'), ('CPRR_score', 'CPRR signature')]:
    g = Mel.obs['group'].astype(str)
    a, b = Mel.obs.loc[g == 'Mattress', score], Mel.obs.loc[g == 'Running', score]
    print(f'{title}: Mattress n = {len(a)}, Running n = {len(b)}; '
          f't-test p = {ttest_ind(a, b)[1]:.3g}, Mann-Whitney U p = {mannwhitneyu(a, b)[1]:.3g}')
    plot_group_violin(Mel.obs[score], g, title)
    plt.show()

## Supplementary Fig. S6F — Signatures increased in melanoma cells from the Running group
AUCell scores (decoupler, `min_n=1`) of melanoma cells, averaged per sample; Running (n = 5) vs Mattress (n = 4),
two-sided Student's t-test of the sample means. Shown are the signatures selected for the figure
(log2FC > 0 and P < 0.05).

In [ ]:
import json
from sc_utils import plot_hbar, signature_group_test

with open(f'{DATA_DIR}/genesets_used.json', 'r') as f:
    genesets = json.load(f)
S6F_TERMS = {'GOBP_COLLAGEN_ACTIVATED_SIGNALING_PATHWAY': 'Collagen-activated signaling pathway',
             'WP_FACTORS_AND_PATHWAYS_AFFECTING_INSULINLIKE_GROWTH_FACTOR_IGF1AKT_SIGNALING':
                 'Factors and pathways affecting insulin-like growth factor (IGF1)-AKT signaling',
             'GOBP_ACTIVATION_OF_INNATE_IMMUNE_RESPONSE': 'Activation of innate-immune response',
             'REACTOME_AKT_PHOSPHORYLATES_TARGETS_IN_THE_NUCLEUS': 'AKT phosphorylates targets in the nucleus',
             'HALLMARK_TGF_BETA_SIGNALING': 'Tgf-beta signaling',
             'REACTOME_INTEGRIN_SIGNALING': 'Integrin-signaling',
             'WP_IL1_SIGNALING_PATHWAY': 'IL1 signaling pathway'}

Mel_sig = adata[adata.obs['Level1_annotation'] == 'Melanoma'].copy()
del Mel_sig.X
Mel_sig = toCount_HVG_Scale(Mel_sig)
res = signature_group_test(Mel_sig, genesets, list(S6F_TERMS)).sort_values('-log10P', ascending=False)
display(res)
fig, ax = plot_hbar(res['-log10P'].values, [S6F_TERMS[k] for k in res.index], color='#3333FF',
                    xlabel='-log(P-value)', figsize=(3, 2.5))
fig.set_label('Melanoma_signatures')
plt.show()